In [ ]:
import pandas as pd
import altair as alt

# STEP 1: Upload CSV in Colab
from google.colab import files
uploaded = files.upload()

# STEP 2: Load CSV
df = pd.read_csv("311_light-requests.csv")

# STEP 3: Drop rows with missing location
df = df.dropna(subset=['Latitude', 'Longitude'])

# STEP 4: Filter to wide bounding box around 125th Street
df_125th = df[
    (df['Latitude'].between(40.808, 40.817)) &
    (df['Longitude'].between(-73.965, -73.925))
]

# STEP 5: Define longitude boundaries for each avenue (west to east)
avenue_bounds = {
    "Broadway":      (-73.963, -73.960),
    "Amsterdam":     (-73.960, -73.957),
    "Morningside":   (-73.957, -73.955),
    "St Nicholas":   (-73.955, -73.952),
    "Lenox":         (-73.952, -73.949),
    "Malcolm X":     (-73.949, -73.946),
    "5th Ave":       (-73.946, -73.943),
    "Madison":       (-73.943, -73.940),
    "Park":          (-73.940, -73.937),
    "Lexington":     (-73.937, -73.934),
    "3rd Ave":       (-73.934, -73.931),
    "2nd Ave":       (-73.931, -73.928)
}

# STEP 6: Assign each complaint to an avenue
def assign_avenue_zone(lon):
    for ave, (min_lon, max_lon) in avenue_bounds.items():
        if min_lon <= lon <= max_lon:
            return ave
    return "Unknown"

df_125th.loc[:, 'Avenue'] = df_125th['Longitude'].apply(assign_avenue_zone)

# STEP 7: Count complaints by avenue
ordered_avenues = list(avenue_bounds.keys())  # ordered west to east
complaint_counts = df_125th['Avenue'].value_counts().reindex(ordered_avenues, fill_value=0).reset_index()
complaint_counts.columns = ['Avenue', 'Complaints']

# STEP 8: Plot with Altair
chart = alt.Chart(complaint_counts).mark_bar().encode(
    x=alt.X('Avenue', sort=ordered_avenues, title='Avenue'),
    y=alt.Y('Complaints', title='Number of Complaints'),
    tooltip=['Avenue', 'Complaints']
).properties(
    title='311 Light Complaints Near 125th Street by Avenue (2024)',
    width=700,
    height=400
).configure_axisX(
    labelAngle=-45
)

chart


Saving 311_light-requests.csv to 311_light-requests (2).csv


<ipython-input-33-c6822f0bc8c6>:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_125th.loc[:, 'Avenue'] = df_125th['Longitude'].apply(assign_avenue_zone)


alt.Chart(...)